In [36]:
# Import pands lib to deal with data as tables
import pandas as pd
# To read vars values that stored in file .env
import os
from dotenv import load_dotenv

# To create connection with DB
from sqlalchemy import create_engine
# check the DB and view some information about them
from sqlalchemy import inspect
load_dotenv=("../.env")
# Set the vars with the responding values
db_user = os.getenv("DB_USER")
db_pass = os.getenv("DB_PASSWORD")
db_name = os.getenv("DB_NAME")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")

engine = create_engine(
f"postgresql+psycopg2://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}"
    )
# Test the connection
with engine.connect() as connection:
    print("successfully connection")

inspector = inspect(engine)
tables = inspector.get_table_names()
tables


successfully connection


['customers',
 'geolocation',
 'order_items',
 'order_payments',
 'order_reviews',
 'orders',
 'products',
 'sellers',
 'product_category_name_translation']

In [ ]:
# read orders table
orders = pd.read_sql("SELECT * FROM orders",engine)
# get first 5 rows
orders.head()
# size of the order table
orders.shape
# information about order table
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  object        
dtypes: datetime64[ns](4), object(4)
memory usage: 6.1+ MB


In [47]:
# create a lest to read all tables
all_tables = ["customers","geolocation","order_items","order_reviews","order_payments","products","sellers","product_category_name_translation","orders"]
# create dir to store all_tables
data ={}
# for loop to read all_tables
for rtable in all_tables:
    data[rtable] = pd.read_sql( f"SELECT * FROM {rtable}",engine)
list(data.keys())
# for loop to ensure num of columns and rows for each table
for rtable,df in data.items():

    print(f"{rtable}:{df.shape}")
# for loop to show each tabel with its columns names as list.
for rtable,df in data.items():
    print(f"\n***{rtable}***")
    print(df.columns.tolist())


customers:(99441, 5)
geolocation:(1000163, 6)
order_items:(112650, 7)
order_reviews:(99224, 7)
order_payments:(103886, 5)
products:(32951, 9)
sellers:(3095, 4)
product_category_name_translation:(71, 2)
orders:(99441, 8)

***customers***
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

***geolocation***
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state', 'geolocation_id']

***order_items***
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

***order_reviews***
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

***order_payments***
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

***products***
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'p

In [51]:
# for loop to test the pk for each table
for rtable,df in data.items():
    print(f"\n****{rtable}****")
    for col in df.columns:
        unique_count = df[col].nunique()
        count_rows = len(df)
        if unique_count == count_rows:
            print(f"the estmated pk is {col}")


****customers****
the estmated pk is customer_id

****geolocation****
the estmated pk is geolocation_id

****order_items****

****order_reviews****

****order_payments****

****products****
the estmated pk is product_id

****sellers****
the estmated pk is seller_id

****product_category_name_translation****
the estmated pk is product_category_name
the estmated pk is product_category_name_english

****orders****
the estmated pk is order_id
the estmated pk is customer_id


In [52]:
# for loop to remove the duplicate
for rtable,dd in data.items():
    dup_rows = dd.duplicated().sum()
    print(f"{rtable}:{dup_rows} duplicated")

customers:0 duplicated
geolocation:0 duplicated
order_items:0 duplicated
order_reviews:0 duplicated
order_payments:0 duplicated
products:0 duplicated
sellers:0 duplicated
product_category_name_translation:0 duplicated
orders:0 duplicated


all the tables has no duplicated rows but to ensure more we make a last list shows all info

In [ ]:
conclosion =[]

# for loop to set the data inside the list ,
for rtable,data_names in data.items():
    #  ass its values as dictionary
    conclosion.append({
        "tables":rtable,
        "rows":len(data_names),
        "columns":len(data_names.columns),
        "dupli":data_names.duplicated().sum()
    })
# confarm the resulte in table view
conclosion_data =pd.DataFrame(conclosion)
conclosion_data

,tables,rows,columns,dupli
0,customers,99441,5,0
1,geolocation,1000163,6,0
2,order_items,112650,7,0
3,order_reviews,99224,7,0
4,order_payments,103886,5,0
5,products,32951,9,0
6,sellers,3095,4,0
7,product_category_name_translation,71,2,0
8,orders,99441,8,0


In [57]:
# to ensure the correct PK for each table
primary_keys = {}
# for loop to set 
for rtable, df in data.items():
    primary_keys[rtable] = []
    for col in df.columns:
        if df[col].nunique() == len(df):
            primary_keys[rtable].append(col)

primary_keys            

{'customers': ['customer_id'],
 'geolocation': ['geolocation_id'],
 'order_items': [],
 'order_reviews': [],
 'order_payments': [],
 'products': ['product_id'],
 'sellers': ['seller_id'],
 'product_category_name_translation': ['product_category_name',
  'product_category_name_english'],
 'orders': ['order_id', 'customer_id']}

In [58]:
for table in ["orders", "order_items", "order_payments", "order_reviews"]:
    df = data[table]
    
    print(f"\n--- {table} ---")
    
    if "order_id" in df.columns:
        print("Rows:", len(df))
        print("Unique order_id:", df["order_id"].nunique())
        print("Duplicated order_id:", df["order_id"].duplicated().sum())


--- orders ---
Rows: 99441
Unique order_id: 99441
Duplicated order_id: 0

--- order_items ---
Rows: 112650
Unique order_id: 98666
Duplicated order_id: 13984

--- order_payments ---
Rows: 103886
Unique order_id: 99440
Duplicated order_id: 4446

--- order_reviews ---
Rows: 99224
Unique order_id: 98673
Duplicated order_id: 551


In [65]:
# check the real PK for all tables
pk_checks ={
    "customers" : ["customer_id"],
    "order_items" : ["order_id" , "order_item_id"],
    "order_payments" : ["order_id","payment_sequential"],
    "order_reviews": ["order_id","review_id"],
    "orders": ["order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    }
# for loop to check the combin is good or not
for rtable,columns in pk_checks.items():
    df = data[rtable]
    dup_count = df.duplicated(subset=columns).sum()
    print(f"{rtable}")
    print(f"pk columns:{columns}")
    print(f"dup combin:{dup_count}")


customers
pk columns:['customer_id']
dup combin:0
order_items
pk columns:['order_id', 'order_item_id']
dup combin:0
order_payments
pk columns:['order_id', 'payment_sequential']
dup combin:0
order_reviews
pk columns:['order_id', 'review_id']
dup combin:0
orders
pk columns:['order_id']
dup combin:0
products
pk columns:['product_id']
dup combin:0
sellers
pk columns:['seller_id']
dup combin:0


In [ ]:
# test the FK between orders and customers
orders = data["orders"]
customers = data["customers"]
# check if there is any missing data
miss_data= ~ orders["customer_id"].isin(customers["customer_id"])
print("number of customer_id missing  is :",miss_data.sum())

number of missing data is : 0


In [72]:
# test the FK between orders and order items
orders = data["orders"]
order_items = data["order_items"]
# check if there is any missing data
miss_data= ~ order_items["order_id"].isin(orders["order_id"])
print("number of order_id missing  is :",miss_data.sum())

number of order_id missing  is : 0


In [ ]:
# test the FK between products and order items
products = data["products"]
order_items = data["order_items"]
# check if there is any missing data
miss_data= ~ order_items["product_id"].isin(products["product_id"])
print("number of product_id missing  is :",miss_data.sum())

number of product_id missing  is : 0


In [77]:
# test the FK between sellers and order items
sellers = data["sellers"]
order_items = data["order_items"]
# check if there is any missing data
miss_data= ~ order_items["seller_id"].isin(sellers["seller_id"])
print("number of seller_id missing  is :",miss_data.sum())

number of seller_id missing  is : 0
